## Tidal Decomposition 

The goal of this notebook is do low pass filter velocity data so we can remove the effects of tides. We will subtract the low-pass filtered velocity data from the in-situ data to get the high frequency component that will have the tidal signals in it. Note if we want to truly isolate tides we should do a harmonic analysis or bandpass filter to more specifically target tidal frequencies. For now, we will just do a low-pass filter. However, these considerations will become important if I want to estimate tidal kinetic energy 

In [ ]:
# imports 
import numpy as np 
import matplotlib.pyplot as plt
import xarray as xr 
import scipy 
import TNB_tools.tides.tidal_decomposition as tides
import TNB_tools.tides.tidal_hydro_relationships as tidal_hydro 

In [ ]:
# load in data - DELL 
# vel = xr.open_dataset("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2018_19 Araon Cruise (ANA09B) - NIWAMooringRetrieval/Data/TNBD/working/tnbd_2018_vel_bot_1093.nc") 

vel = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_vel_bot_1093.nc')

In [ ]:
# load in CTD data 
# ctd = xr.open_dataset("//thepenguin/penguin2/Data/TNB/mooring/LDEO/2018_19 Araon Cruise (ANA09B) - NIWAMooringRetrieval/Data/TNBD/working/tnbd_2018_ctd_bot_1093.nc")

ctd = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_tools/MastersThesis/tnbd_2018_ctd_bot_1093.nc')

In [ ]:
ctd_15 = ctd.resample(time='15min').mean()

In [ ]:
# select time range to match with velocity data 
ctd_15 = ctd_15.sel(time=slice('2018-03-14','2018-12-31'))

In [ ]:
# select time range to match with CTD data
vel = vel.sel(time=slice('2018-03-14','2018-12-31'))

In [ ]:
sampling_hours = 0.25  # sampling period (hrs)
window_days = 2     # desired window 
window_size = int(window_days * 24 / sampling_hours)   # window size based on sampling period 


In [ ]:
window_size

In [ ]:
if window_size % 2 == 0:
    window_size += 1

In [ ]:
from scipy.signal import windows
import numpy as np

cosine_window = windows.cosine(window_size)
cosine_window /= cosine_window.sum()  # normalize so it acts like a moving average


In [ ]:
len(cosine_window)

In [ ]:
sampling_hours = 0.25  # sampling period (hrs)
window_days = 1     # desired window 
window_size = int(window_days * 24 / sampling_hours) 

if window_size % 2 == 0:
    window_size += 1

cosine_window = windows.cosine(window_size)
cosine_window /= cosine_window.sum()  # normalize so it acts like a moving average

# assume velocity_da is your DataArray with dimension "time"
cosine_weights = xr.DataArray(cosine_window, dims="window")

# create rolling window and apply weights
tnbd_east_filtered = (
    vel.u
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)

tnbd_north_filtered = (
    vel.v
    .rolling(time=window_size, center=True)
    .construct("window")
    .dot(cosine_weights)
)


In [ ]:
tnbd_east_filtered.plot()

In [ ]:
# subtract low-pass from in-situ 
tides_zonal = vel.u - tnbd_east_filtered
tides_meridional = vel.v - tnbd_north_filtered

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tnbd_east_tidal)
plt.plot(vel.u)
plt.plot(tnbd_east_filtered)

In [ ]:
# using functions 
lowpass_ds, highpass_ds = tides.decompose_tidal_dataset(
    vel, var_names=["u", "v"], window_days=3, sampling_hours=0.25
)

u_lp = lowpass_ds["u"]
u_tides = highpass_ds["u"]

v_lp = lowpass_ds["v"]
v_tides = highpass_ds["v"]

In [ ]:
%matplotlib widget

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(u_tides.time,u_tides,label='u Tides')
plt.plot(vel.time,vel.u,label='u in-situ')
plt.plot(u_lp.time,u_lp,label='u 3-day lowpass')
plt.plot(u_tides.time,(u_tides + u_lp))
plt.legend()
plt.ylabel('u velocity')

In [ ]:
ctd_monthly = ctd_15.resample(time = '1MS').mean()

In [ ]:
tides = np.sqrt(u_tides**2 + v_tides**2)

In [ ]:
tides

In [ ]:
temp_mean = np.mean(ctd_15.temp)

In [ ]:
temp_mean

In [ ]:
temp_anom = ctd_15.temp-temp_mean

In [ ]:
rolling = 288 #three day rolling average
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(tides.rolling(time=rolling).mean().time,tides.rolling(time=rolling).mean())
ax2 = ax.twinx()
ax2.plot(ctd_15.rolling(time=rolling).mean().time,ctd_15.rolling(time=rolling).mean().temp,color='tab:red')

In [ ]:
rolling = 672 #weekly day rolling average
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(tides.rolling(time=rolling).mean().time,tides.rolling(time=rolling).mean())
ax2 = ax.twinx()
ax2.plot(ctd_15.rolling(time=rolling).mean().time,(ctd_15.rolling(time=rolling).mean().abs_sal-ctd_15.rolling(time=rolling).mean().abs_sal.mean()),color='tab:red')

In [ ]:
tides_monthly = tides.resample(time='1MS').mean()

In [ ]:
tides_monthly

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,4))
line1, = ax1.plot(tides_monthly.time,tides_monthly,color='k',label='tides')
ax1.set_ylabel('Tides (cm/s)')

ax2 = ax1.twinx()
line2, = ax2.plot(ctd_monthly.time,ctd_monthly.temp,color='tab:red',label='temp')
ax2.tick_params(axis='y', colors='tab:red')
ax2.set_ylabel('Temp')

# 6. Combine handles and labels from both axes into one legend
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)


In [ ]:
fig, ax1 = plt.subplots(figsize=(12,4))
line1, = ax1.plot(tides_monthly.time,tides_monthly,color='k',label='tides')
ax1.set_ylabel('Tides (cm/s)')

ax2 = ax1.twinx()
line2, = ax2.plot(ctd_monthly.time,ctd_monthly.abs_sal,color='tab:blue',label='sal')
ax2.tick_params(axis='y', colors='tab:blue')
ax2.set_ylabel('Absolute Salinity')

# 6. Combine handles and labels from both axes into one legend
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)

In [ ]:
flow = np.sqrt(u_lp**2 + v_lp**2)

In [ ]:
flow_monthly = flow.resample(time='1MS').mean()

In [ ]:
vel_monthly = vel.resample(time='1MS').mean()

In [ ]:
speed_monthly = np.sqrt(vel_monthly.u**2 + vel_monthly.v**2)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tides_monthly.time,tides_monthly,label='tides')
plt.plot(flow_monthly.time,flow_monthly,label='low-pass')
plt.plot(speed_monthly.time,speed_monthly,label='speed')

plt.legend()

In [ ]:
ctd_monthly

In [ ]:
lowpass_ds

In [ ]:
highpass_ds

### Tides w hydrographic signals 

In [ ]:
_, u_hp = tides.decompose_tidal(vel.u, window_days=1, sampling_hours=0.25)
_, v_hp = tides.decompose_tidal(vel.v, window_days=1, sampling_hours=0.25)

In [ ]:
tke_envelope = tidal_hydro.tidal_energy_envelope(u_hp, v_hp,window_days=14.8,sampling_hours=0.25)

In [ ]:
ctd_15

In [ ]:
s_lowpass, s_anomaly = tidal_hydro.lowpass_and_anomaly(ctd_15.sigma0,window_days=1,sampling_hours=0.25)

In [ ]:
lags, corr, n_eff, conf95 = tidal_hydro.lagged_cross_correlation(tke_envelope, s_anomaly, max_lag=60)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Salinity: low-pass state and anomaly
axes[0].plot(s_lowpass.time, s_lowpass, color="k", lw=1.5, label="S low-pass")
# axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("Salinity (psu)")
axes[0].legend(loc="upper right")

ax0b = axes[0].twinx()
ax0b.plot(s_anomaly.time, s_anomaly, color="tab:blue", lw=1, alpha=0.7, label="S anomaly")
ax0b.axhline(0, color="tab:blue", lw=0.5, ls="--")
ax0b.set_ylabel("Salinity anomaly (psu)", color="tab:blue")
ax0b.tick_params(axis="y", labelcolor="tab:blue")

# Tidal energy envelope
axes[1].plot(tke_envelope.time, tke_envelope, color="tab:red", lw=1.5)
axes[1].set_ylabel("Tidal energy\nenvelope (m²/s²)", color="tab:red")
axes[1].tick_params(axis="y", labelcolor="tab:red")
axes[1].set_xlabel("Time")

axes[0].set_title("Salinity vs. tidal energy envelope")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lags, corr, color="k")
ax.axhline(conf95, color="gray", ls="--", lw=1, label=f"95% CI (N_eff={n_eff:.1f})")
ax.axhline(-conf95, color="gray", ls="--", lw=1)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("Lag (samples)")
ax.set_ylabel("Correlation")
ax.legend()
plt.show()